# Train Ibiko's adapter

This notebook turns what Ibiko knows into a **LoRA adapter** for the model on your device (Qwen3 1.7B, the default
or the Fast one: both are the same weights, so one adapter fits both).

1. In Ibiko: **Settings › Usage › Export**. It saves `Downloads/Ibiko/ibiko-training-<date>.jsonl`.
2. Here: **Runtime › Change runtime type › T4 GPU**, then **Runtime › Run all**. It asks for the file in step 2.
3. At the end it downloads `ibiko.adapter.gguf`. In Ibiko: **Settings › Usage › Install adapter** and pick it.
   It is used from the model's next start. Holding the button puts the previous adapter back.

About 10 to 30 minutes on a free T4 for a few hundred examples. Nothing here is sent anywhere but Colab and
Hugging Face (to download the base model and the converter).

## 1. Check the GPU and install

In [ ]:
import subprocess, sys
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
print(gpu.stdout.strip() or "No GPU: Runtime > Change runtime type > T4 GPU, then run again.")
assert gpu.returncode == 0, "This notebook needs a GPU runtime."

In [ ]:
!pip -q install "transformers>=4.51,<5" "peft>=0.15,<1" "datasets>=3,<5" "accelerate>=1.6" "huggingface_hub>=0.30"

## 2. Settings and your training file

The defaults suit a few hundred examples. More examples: keep `EPOCHS` at 1 or 2. Very few (under 50): 3.

In [ ]:
BASE = "Qwen/Qwen3-1.7B"      # the weights of Ibiko's device model (default and Fast)
EPOCHS = 2
RANK = 16                     # the adapter's size: 16 is about 35 MB as f16
LEARNING_RATE = 2e-4
MAX_TOKENS = 1024             # longer examples are cut at the end
LLAMA_CPP_TAG = "b12319"      # the converter's version (llama.cpp), pinned
OUT_NAME = "ibiko.adapter.gguf"

In [ ]:
import glob, os
try:
    from google.colab import files
    uploaded = files.upload()          # pick ibiko-training-<date>.jsonl
    DATA = next(iter(uploaded))
except ImportError:                    # not in Colab: the newest export in this folder
    DATA = sorted(glob.glob("ibiko-training-*.jsonl"))[-1]
print("Training file:", DATA, os.path.getsize(DATA), "bytes")

## 3. Read the examples

Each line is `{"messages": [...]}` ending with Ibiko's answer. Lines that are broken, empty or a failure
("couldn't reach", "rate limit") are dropped, and repeats are kept once.

In [ ]:
import json, random, re

FAILURE = re.compile(r"couldn'?t reach|could not reach|timed? ?out|connection (refused|reset|failed)|rate limit|"
                     r"api key|server error|service unavailable|i'?m offline|no internet", re.I)

def clean(path):
    seen, out = set(), []
    for line in open(path, encoding="utf-8"):
        try:
            msgs = json.loads(line)["messages"]
        except Exception:
            continue
        msgs = [m for m in msgs if isinstance(m, dict) and m.get("role") in ("system", "user", "assistant")
                and isinstance(m.get("content"), str) and m["content"].strip()]
        if len(msgs) < 2 or msgs[-1]["role"] != "assistant" or msgs[-2]["role"] != "user":
            continue
        if FAILURE.search(msgs[-1]["content"]):
            continue
        key = (msgs[-2]["content"].strip().lower(), msgs[-1]["content"].strip())
        if key in seen:
            continue
        seen.add(key)
        out.append(msgs)
    return out

examples = clean(DATA)
random.seed(7)
random.shuffle(examples)
held = examples[: max(1, min(20, len(examples) // 20))] if len(examples) >= 20 else []
train = examples[len(held):]
print(len(examples), "examples:", len(train), "to train,", len(held), "held back to check")
assert len(train) >= 5, "Too few examples: use Ibiko longer, check the answers you agree with, then export again."

## 4. Load the base model

The full-precision weights are trained, not the 4-bit file on the device: the adapter made here is then laid over the
device's 4-bit copy of these same weights.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
tok = AutoTokenizer.from_pretrained(BASE)
model = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=DTYPE, device_map="cuda")
model.gradient_checkpointing_enable()
model.config.use_cache = False
print("Loaded", BASE, "as", DTYPE)

## 5. Turn the examples into training text

Each example is written the way Ibiko's engine asks the model (thinking off), and only Ibiko's answer is learned:
the question and instructions are context, not something to copy.

In [ ]:
END = "<|im_end|>"

def prompt_text(msgs):
    return tok.apply_chat_template(msgs[:-1], tokenize=False, add_generation_prompt=True, enable_thinking=False)

def encode(msgs):
    p = tok(prompt_text(msgs), add_special_tokens=False)["input_ids"]
    a = tok(msgs[-1]["content"].strip() + END, add_special_tokens=False)["input_ids"]
    ids = (p + a)[:MAX_TOKENS]
    labels = ([-100] * len(p) + a)[:MAX_TOKENS]
    return {"input_ids": ids, "labels": labels}

from datasets import Dataset
train_ds = Dataset.from_list([encode(m) for m in train])
train_ds = train_ds.filter(lambda e: any(l != -100 for l in e["labels"]))   # an answer cut away entirely teaches nothing
print(len(train_ds), "training sequences; longest", max(len(e["input_ids"]) for e in train_ds), "tokens")

def collate(batch):
    n = max(len(e["input_ids"]) for e in batch)
    pad = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id
    ids = torch.full((len(batch), n), pad)
    labels = torch.full((len(batch), n), -100)
    mask = torch.zeros((len(batch), n), dtype=torch.long)
    for i, e in enumerate(batch):
        k = len(e["input_ids"])
        ids[i, :k] = torch.tensor(e["input_ids"])
        labels[i, :k] = torch.tensor(e["labels"])
        mask[i, :k] = 1
    return {"input_ids": ids, "labels": labels, "attention_mask": mask}

## 6. What it answers before training (held-back questions)

In [ ]:
def ask(msgs, adapter_on=True):
    text = prompt_text(msgs)
    ids = tok(text, return_tensors="pt").to(model.device)
    model.config.use_cache = True
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=120, do_sample=False)
    model.config.use_cache = False
    return tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()

before = {i: ask(m) for i, m in enumerate(held[:5])}
for i, m in enumerate(held[:5]):
    print("Q:", m[-2]["content"][:200]); print("  before:", before[i][:300]); print()

## 7. Train the adapter

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import Trainer, TrainingArguments

model.enable_input_require_grads()
model = get_peft_model(model, LoraConfig(r=RANK, lora_alpha=RANK * 2, lora_dropout=0.05, bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
model.print_trainable_parameters()

args = TrainingArguments(output_dir="out", num_train_epochs=EPOCHS, learning_rate=LEARNING_RATE,
        per_device_train_batch_size=2, gradient_accumulation_steps=4, lr_scheduler_type="cosine",
        warmup_ratio=0.05, logging_steps=5, save_strategy="no", report_to=[],
        bf16=DTYPE == torch.bfloat16, fp16=DTYPE == torch.float16, remove_unused_columns=False)
Trainer(model=model, args=args, train_dataset=train_ds, data_collator=collate).train()
model.save_pretrained("ibiko-lora")
print("Adapter saved:", os.listdir("ibiko-lora"))

## 8. Compare: before and after (held-back questions, never trained on)

In [ ]:
for i, m in enumerate(held[:5]):
    print("Q:", m[-2]["content"][:200])
    print("  kept answer:", m[-1]["content"][:300])
    print("  before:     ", before[i][:300])
    print("  after:      ", ask(m)[:300]); print()

## 9. Convert to a GGUF adapter for Ibiko's engine

llama.cpp's own converter, at a pinned version, writes the adapter in the format the engine on the device reads.

In [ ]:
from huggingface_hub import snapshot_download
base_dir = snapshot_download(BASE, allow_patterns=["*.json", "*.txt", "tokenizer*", "*.model"])
if not os.path.isdir("llama.cpp"):
    !git clone -q --depth 1 --branch {LLAMA_CPP_TAG} https://github.com/ggml-org/llama.cpp
!pip -q install ./llama.cpp/gguf-py "sentencepiece" "protobuf"
!python llama.cpp/convert_lora_to_gguf.py ibiko-lora --base {base_dir} --outtype f16 --outfile {OUT_NAME}
assert os.path.isfile(OUT_NAME), "The conversion failed: see the lines above."
with open(OUT_NAME, "rb") as f:
    assert f.read(4) == b"GGUF"
print(OUT_NAME, round(os.path.getsize(OUT_NAME) / 1e6, 1), "MB")

## 10. Download it, then install it in Ibiko

In [ ]:
try:
    from google.colab import files
    files.download(OUT_NAME)
except ImportError:
    print("Saved as", os.path.abspath(OUT_NAME))
print("In Ibiko: Settings > Usage > Install adapter, pick", OUT_NAME, "- used from the model's next start.")